# Genie Cache & Queue — Demo

A API do Genie tem um hard limit de **5 chamadas por minuto por workspace**.
Este notebook demonstra o problema e como o app resolve com cache semantico, retry e fila.

## Cenarios:
1. **Sem o app**: 7 chamadas diretas ao Genie → ultimas recebem **HTTP 429**
2. **Com o app (1a rodada)**: Mesmas 7 queries via app → todas completam
3. **Com o app (2a rodada)**: Mesmas queries de novo → **todas do cache em <0.1s**

O app e um **drop-in replacement** — mesmos endpoints, mesmos payloads, mesmas respostas.
Basta trocar a URL base.

## Setup

In [ ]:
import requests
import time
import json
from databricks.sdk import WorkspaceClient

WORKSPACE_HOST = "https://fevm-genie-cache-test.cloud.databricks.com"
APP_HOST = "https://genie-cache-queue-7474650836156271.aws.databricksapps.com"
SPACE_ID = "01f11f1ae00114379e7671c8a4b8459f"
WAREHOUSE_ID = "20cbd6750a2ef9ce"

# Auth: SDK gera OAuth headers que funcionam tanto para Genie direto quanto para o App
w = WorkspaceClient()
auth_headers = w.config.authenticate()
headers = {**auth_headers, "Content-Type": "application/json"}

questions = [
    "How many customers are there?",
    "What is the total revenue by region?",
    "Top 5 suppliers by order count",
    "Average order value per year",
    "How many parts are in the catalog?",
    "Total revenue for EUROPE region",
    "Number of orders in 1995",
]

def extract_response(data):
    """Extract SQL, text and status from Genie response."""
    sql, text = None, None
    for att in data.get("attachments", []):
        if isinstance(att, dict):
            if "query" in att and att["query"]:
                sql = att["query"].get("query", "")
            if "text" in att and att["text"]:
                t = att["text"].get("content", "")
                if t and "semantic cache" not in t:  # skip cache notice
                    text = t
    if not sql:
        sql = data.get("sql_query")
    return sql, text

def call_genie(host, question):
    """Call start-conversation and return (http_status, data, elapsed)."""
    start = time.time()
    resp = requests.post(
        f"{host}/api/2.0/genie/spaces/{SPACE_ID}/start-conversation",
        headers=headers, json={"content": question}, timeout=180,
    )
    elapsed = time.time() - start
    data = resp.json() if resp.text and resp.text.strip() else {}
    return resp.status_code, data, elapsed

# Verificar autenticacao
test = requests.get(f"{APP_HOST}/api/v1/health", headers=headers)
print(f"App health: {test.status_code} {test.text[:80]}")
test2 = requests.get(f"{APP_HOST}/api/v1/debug/headers", headers=headers)
debug = test2.json() if test2.status_code == 200 else {}
print(f"App recebe token: {'x-forwarded-access-token' in debug.get('headers', {})}")
print(f"App recebe email: {debug.get('headers', {}).get('x-forwarded-email', 'N/A')}")
print(f"\nQuestions: {len(questions)}")

## Cenario 1: Chamadas diretas ao Genie

7 queries diretas. Limite: **5/min por workspace**. As ultimas devem dar **429**.

In [ ]:
print("=" * 90)
print("CENARIO 1: Chamadas DIRETAS ao Genie")
print("=" * 90)

direct_results = []
for i, q in enumerate(questions, 1):
    http, data, elapsed = call_genie(WORKSPACE_HOST, q)
    sql, text = extract_response(data)

    if http == 429:
        retry = data.get("retry_after", "?")
        print(f"[{i}/7] 429 RATE LIMITED | {elapsed:.1f}s")
        print(f"       Query:     {q}")
        print(f"       Resultado: BLOQUEADO")
    elif http == 200 and sql:
        print(f"[{i}/7] 200 OK | {elapsed:.1f}s")
        print(f"       Query:     {q}")
        print(f"       SQL:       {sql[:100]}")
        if text:
            print(f"       Resposta:  {text[:140]}")
    else:
        print(f"[{i}/7] {http} | {elapsed:.1f}s")
        print(f"       Query:     {q}")
        print(f"       Raw:       {json.dumps(data)[:150]}")

    direct_results.append({"q": q, "http": http, "time": elapsed, "sql": sql, "text": text})
    print()

ok = sum(1 for r in direct_results if r["http"] == 200 and r["sql"])
blocked = sum(1 for r in direct_results if r["http"] == 429)
print(f"RESULTADO: {ok} OK com SQL, {blocked} bloqueadas por rate limit")

## Cenario 2: Via App

Mesmas 7 queries via app. Endpoint **identico** — so muda a URL base.

### Configurar app

In [ ]:
resp = requests.put(
    f"{APP_HOST}/api/v1/config", headers=headers,
    json={"genie_space_id": SPACE_ID, "sql_warehouse_id": WAREHOUSE_ID,
          "similarity_threshold": 0.92, "cache_ttl_seconds": 86400},
)
print(json.dumps(resp.json(), indent=2))

### 2a. Primeira rodada (popula cache)

Queries novas. App chama Genie, gerencia rate limit, faz retry. **Caller nunca ve 429.**

In [ ]:
print("=" * 90)
print("CENARIO 2a: Via APP — primeira rodada")
print("=" * 90)

app_results_1 = []
for i, q in enumerate(questions, 1):
    http, data, elapsed = call_genie(APP_HOST, q)
    sql, text = extract_response(data)
    conv_id = data.get("conversation_id", "")
    from_cache = conv_id.startswith("ccache_")
    status = data.get("status", "?")

    source = "CACHE" if from_cache else "GENIE"
    print(f"[{i}/7] {status} | {source} | {elapsed:.1f}s")
    print(f"       Query:     {q}")
    if sql:
        print(f"       SQL:       {sql[:100]}")
    if text:
        print(f"       Resposta:  {text[:140]}")
    if not sql and not text:
        print(f"       Raw:       {json.dumps(data)[:200]}")
    print()

    app_results_1.append({"q": q, "time": elapsed, "cache": from_cache, "sql": sql, "text": text, "status": status})

genie = sum(1 for r in app_results_1 if not r["cache"] and r["sql"])
cached = sum(1 for r in app_results_1 if r["cache"])
print(f"RESULTADO: {genie} Genie calls, {cached} cache hits")

### 2b. Segunda rodada (cache hit)

Mesmas queries — todas do cache. **SQL identico**, resposta em <0.1s.

In [ ]:
print("=" * 90)
print("CENARIO 2b: Via APP — segunda rodada (cache)")
print("=" * 90)

app_results_2 = []
for i, q in enumerate(questions, 1):
    http, data, elapsed = call_genie(APP_HOST, q)
    sql, text = extract_response(data)
    conv_id = data.get("conversation_id", "")
    from_cache = conv_id.startswith("ccache_")

    prev = app_results_1[i-1]["sql"] if i-1 < len(app_results_1) else None
    match = "MATCH" if sql and prev and sql.strip() == prev.strip() else "-"

    print(f"[{i}/7] CACHE HIT | {elapsed:.3f}s | SQL {match}")
    print(f"       Query: {q}")
    if sql:
        print(f"       SQL:   {sql[:100]}")
    print()

    app_results_2.append({"q": q, "time": elapsed, "cache": from_cache, "sql": sql})

total = sum(r["time"] for r in app_results_2)
all_cached = all(r["cache"] for r in app_results_2)
print(f"RESULTADO: 7/7 cache hits | Total: {total:.2f}s | Media: {total/7:.3f}s | Todas cache: {all_cached}")

### Estado do cache

In [ ]:
resp = requests.get(f"{APP_HOST}/api/v1/cache", headers=headers)
entries = resp.json() if resp.status_code == 200 else []
print(f"Cache entries: {len(entries)}\n")
for e in entries:
    print(f"  Query:   {e['query_text'][:60]}")
    print(f"  SQL:     {e['sql_query'][:80]}")
    print(f"  Uses:    {e['use_count']}  |  Created: {e['created_at']}")
    print()

## Comparacao Final

In [ ]:
print("=" * 95)
print(f"{'Query':30s} | {'Direto':>14s} | {'App 1a':>16s} | {'App 2a':>14s}")
print("-" * 95)

for i, q in enumerate(questions):
    d = direct_results[i] if i < len(direct_results) else {"http": "?", "time": 0, "sql": None}
    a1 = app_results_1[i] if i < len(app_results_1) else {"time": 0, "cache": False, "sql": None}
    a2 = app_results_2[i] if i < len(app_results_2) else {"time": 0, "cache": False}

    d_col = "429 BLOCKED" if d["http"] == 429 else (f"{d['time']:.1f}s OK" if d["sql"] else f"{d['http']}")
    a1_col = f"{a1['time']:.1f}s CACHE" if a1.get("cache") else (f"{a1['time']:.1f}s GENIE" if a1.get("sql") else f"{a1['time']:.1f}s ?")
    a2_col = f"{a2['time']:.2f}s CACHE"

    print(f"  {q[:28]:28s} | {d_col:>14s} | {a1_col:>16s} | {a2_col:>14s}")

print()
print("Direto:  queries 6-7 bloqueadas pelo rate limit (429)")
print("App 1a:  todas completam — retry automatico, sem 429 para o caller")
print("App 2a:  todas do cache — <0.1s, zero chamadas ao Genie, mesmo SQL")